# Executing prompts programmatically

## Minimal prompt execution

In [1]:
from openai import OpenAI
import getpass

In [ ]:
# OPENAI_API_KEY = getpass.getpass('Enter your OPENAI_API_KEY')
OPENAI_API_KEY = ''


In [3]:
client = OpenAI(api_key=OPENAI_API_KEY)

In [4]:
prompt_input = """Write a concise message to remind
users to be vigilant about phishing attacks."""
response = client.chat.completions.create(
model="gpt-5-nano",
messages=[
{"role": "system", "content": "You are a helpful assistant."},
{"role": "user", "content": prompt_input}
]
)
print(response)

ChatCompletion(id='chatcmpl-DRwzqGxTHUzKg4LSk1uJjLH5Ehvjk', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='Reminder: Be vigilant against phishing. Verify the sender, hover over links before clicking, never share passwords or sensitive data, and report suspicious messages to IT.', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None))], created=1775554010, model='gpt-5-nano-2025-08-07', object='chat.completion', service_tier='default', system_fingerprint=None, usage=CompletionUsage(completion_tokens=552, prompt_tokens=31, total_tokens=583, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=512, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0)))


## Running prompts with LangChain

In [5]:
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(openai_api_key=OPENAI_API_KEY,
model_name="gpt-5-nano")

In [6]:
prompt_input = """Write a concise message to remind
users to be vigilant about phishing attacks."""

In [7]:
response = llm.invoke(prompt_input)
print(response.content)

Reminder: Be vigilant against phishing. Verify the sender, hover over links to check URLs, never share passwords, and report suspicious emails immediately.


## Prompt templates

In [10]:
def generate_text_summary_prompt(text, num_words, tone):
    return f"You are an experienced copywriter. Write a {num_words} words summary of the following text, using a {tone} tone: {text}"

In [11]:
segovia_aqueduct_text = """The Aqueduct of Segovia (Spanish:
Acueducto de Segovia) is a Roman aqueduct in Segovia, Spain.
It was built around the first century AD to channel water from
springs in the mountains 17 kilometres (11 mi) away to the
city's fountains, public baths and private houses, and was in
use until 1973.
Its elevated section, with its complete arcade of 167 arches,
is one of the best-preserved Roman aqueduct bridges and the
foremost symbol of Segovia, as evidenced by its presence on the
city's coat of arms.
The Old Town of Segovia and the aqueduct, were declared a UNESCO
World Heritage Site in 1985. As the aqueduct lacks a legible
inscription (one was apparently located in the structure's attic,
or top portion[citation needed]), the date of construction cannot be
definitively determined. The general date of the Aqueduct's
construction was long a mystery, although it was thought to have
been during the 1st century AD, during the reigns of the Emperors
Domitian, Nerva, and Trajan. At the end of the 20th century,
Géza Alföldy deciphered the text on the dedication plaque by
studying the anchors that held the now missing bronze letters
in place. He determined that Emperor Domitian (AD 81–96) ordered
its construction[1] and the year 98 AD was proposed as the most
likely date of completion.[2] However, in 2016 archeological
evidence was published which points to a slightly later date,
after 112 AD, during the government of Trajan or in the
beginning of the government of emperor Hadrian,
from 117 AD."""

In [15]:
input_prompt = generate_text_summary_prompt(text=segovia_aqueduct_text, num_words=100, tone="knowledgebale and engaging")
response = llm.invoke(input_prompt)
print(response.content)

The Aqueduct of Segovia is a jewel in Segovia, Spain. Built in first century AD to channel mountain springs 17 kilometres away to fountains, baths and houses, it remained in use until 1973. Its elevated section, with 167 arches, is among the best-preserved Roman aqueducts and serves as Segovia’s emblem, even appearing on the coat of arms. In 1985, Segovia and its aqueduct were designated UNESCO World Heritage. Lacking a legible inscription, the exact date is uncertain. Alföldy dated it to 98 AD under Domitian, while 2016 evidence suggests after 112 AD under Trajan or Hadrian, and the mystery endures.


### Using LangChain’s PromptTemplate

In [18]:
from langchain_core.prompts import PromptTemplate


In [19]:
prompt_template = PromptTemplate.from_template("""You are an experienced copywriter.
Write a {num_words} words summary of the following text,
using a {tone} tone: {text}""")

In [20]:
prompt = prompt_template.format(text=segovia_aqueduct_text, num_words=20, tone="knowledgebale and engaging")

In [22]:
response = llm.invoke(prompt)
print(response.content)

Roman aqueduct of Segovia, built 1st century AD to carry water 17 km from mountains; 167 arches, UNESCO symbol, iconic.


### Implementing few-shot learning with  LangChain

In [23]:
from langchain_openai import ChatOpenAI

In [25]:
llm = ChatOpenAI(openai_api_key=OPENAI_API_KEY, model_name="gpt-5-mini")

In [26]:
prompt_input = """Classify the following numbers as Abra, Kadabra
➥or Abra Kadabra:
3, 4, 5, 7, 8, 10, 11, 13, 35
Examples:
6 // not divisible by 5, not divisible by 7 // None
15 // divisible by 5, not divisible by 7 // Abra
12 // not divisible by 5, not divisible by 7 // None
21 // not divisible by 5, divisible by 7 // Kadabra
70 // divisible by 5, divisible by 7 // Abra Kadabra
"""

In [28]:
response = llm.invoke(prompt_input)
print(response.content)

3  // not divisible by 5, not divisible by 7 // None
4  // not divisible by 5, not divisible by 7 // None
5  // divisible by 5, not divisible by 7 // Abra
7  // not divisible by 5, divisible by 7 // Kadabra
8  // not divisible by 5, not divisible by 7 // None
10 // divisible by 5, not divisible by 7 // Abra
11 // not divisible by 5, not divisible by 7 // None
13 // not divisible by 5, not divisible by 7 // None
35 // divisible by 5, divisible by 7 // Abra Kadabra


In [31]:
from langchain_core.prompts.few_shot import FewShotPromptTemplate
from langchain_core.prompts.prompt import PromptTemplate

In [32]:
examples = [
{
"number": 6,
"reasoning": "not divisible by 5 nor by 7",
"result": "None"
},
{
"number": 15,
"reasoning": "divisible by 5 but not by 7",
"result": "Abra"
},
{
"number": 12,
"reasoning": "not divisible by 5 nor by 7",
"result": "None"
},
{
"number": 21,
"reasoning": "divisible by 7 but not by 5",
"result": "Kadabra"
},
{
"number": 70,
"reasoning": "divisible by 5 and by 7",
"result": "Abra Kadabra"
} ]

In [33]:
example_prompt = PromptTemplate(input_variables=["number", "reasoning", "result"], template = "{number} \\ {reasoning} \\ {result}")
few_shot_prompt = FewShotPromptTemplate(examples=examples, 
                                        example_prompt=example_prompt, 
                                        suffix="Classify the following numbers as Abra, Kadabra or Abra Kadabra:{comma_delimited_input_numbers}",
                                        input_variables=["comma_delimited_input_numbers"]
                                       )

In [34]:
prompt_input  = few_shot_prompt.format(
    comma_delimited_input_numbers="3, 4, 5, 7, 8, 10, 11, 13, 35"
)

response = llm.invoke(prompt_input)
print(response.content)

Rule: divisible by 5 → Abra; divisible by 7 → Kadabra; divisible by both → Abra Kadabra; otherwise → None.

3 → None
4 → None
5 → Abra
7 → Kadabra
8 → None
10 → Abra
11 → None
13 → None
35 → Abra Kadabra
